# YOLO Training

## GPU availability

In [1]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"Torch Version: {torch.__version__}")

# 1. Prüfen ob CUDA generell verfügbar ist
if not torch.cuda.is_available():
    print(" KEINE GPU GEFUNDEN! Torch läuft auf CPU.")
else:
    try:
        # 2. Treiber-Status abfragen (System-Level)
        print("\n--- System GPU Status (nvidia-smi) ---")
        !nvidia-smi
        
        # 3. Harter Test: Eine Tensor-Berechnung auf der GPU durchführen
        # Das entlarvt den 'Suspend-Bug', bei dem is_available() True sagt, aber nichts geht.
        x = torch.tensor([1.0, 2.0]).cuda()
        print(f"\n GPU TEST ERFOLGREICH: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
        
    except Exception as e:
        print(f"\n GPU FEHLER: Der Treiber scheint zu hängen (Suspend Bug?).")
        print(f"Fehler: {e}")
        print("Lösung: PC Neustarten.")

Python Version: 3.12.3 (main, Nov  6 2025, 13:44:16) [GCC 13.3.0]
Torch Version: 2.9.1+cu128

--- System GPU Status (nvidia-smi) ---
Fri Jan  2 15:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.195.03             Driver Version: 570.195.03     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:2D:00.0 Off |                  N/A |
|  0%   37C    P8             15W /  350W |     166MiB /  24576MiB |      0%      Default |
|      

## Training YOLO on all classes

### YOLO run on the original (not stratified, not oversampled) dataset

In [ ]:
from ultralytics import YOLO

# Wähle das Modell:
# 'yolo11n.pt' -> Nano (Schnellster Test)
# 'yolo11m.pt' -> Medium (Gute Balance für später)
# 'yolo11l.pt' -> Large (Hohe Genauigkeit für 3090)
MODEL_VARIANT = 'yolo11n.pt' 

print(f"Lade Modell: {MODEL_VARIANT}...")
model = YOLO(MODEL_VARIANT)

# Training starten
results = model.train(
    data='../../dataset/04_all_classes/data.yaml', # Pfad muss stimmen!
    epochs=50,          # Für Nano Test reichen 50. Für Large später 100+
    imgsz=640,          # Standard. Bei 3090 könntest du auch 1280 testen (High Res)
    batch=32,           # 3090 packt bei Nano locker 64/128, bei Large ca 16/24
    name='compound_yolo11n_sim2real',
    device=0,           # Erzwingt GPU 0
    workers=8,          # Schnelleres Laden der Daten
    exist_ok=True       # Überschreibt den Ordner falls er existiert (gut zum Testen)
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/tobiasponeschtu/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Lade Modell: yolo11n.pt...
Ultralytics 8.3.240 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015

### YOLO run on the new, stratified and oversampled Dataset

In [ ]:
from ultralytics import YOLO

# --- KONFIGURATION ---
# Für den ersten schnellen Pipeline-Check ist Nano super.
# Sobald das läuft -> Auf 'yolo11m.pt' oder 'yolo11l.pt' wechseln!
MODEL_VARIANT = 'yolo11n.pt' 

print(f"Lade Modell: {MODEL_VARIANT}...")
model = YOLO(MODEL_VARIANT)

# Training starten
results = model.train(
    data='../../dataset/04_all_classes/data.yaml', 
    epochs=50,          
    
    # TIPP FÜR SCIENTIFIC FIGURES:
    # 640px ist oft zu klein für feine Achsenbeschriftungen.
    # Da du eine 3090 hast, probier direkt 1024 oder 1280, 
    # das hilft massiv bei kleinen Subpanels!
    imgsz=640,          # Für den Nano-Schnelltest ok. Später: 1280
    
    # TIPP FÜR RTX 3090:
    # Bei Nano kannst du hier locker 128 oder 256 nehmen.
    # Bei Medium/Large sind 16-32 realistisch.
    batch=64,           
    
    name='compound_yolo11n_sim2real_v1',
    device=0,           
    workers=8,
    
    # WICHTIG: Plots werden gespeichert, damit wir die Confusion Matrix sehen
    plots=True,  
)

Lade Modell: yolo11n.pt...
New https://pypi.org/project/ultralytics/8.3.241 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.240 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=

### YOLO run with better image resolution

In [ ]:
from ultralytics import YOLO

# 1. Small Modell (S, nicht N oder M)
# Bietet mehr Kapazität für Details als Nano, trainiert aber flotter als Medium.
model = YOLO('yolo11s.pt') 

# 2. Training
results = model.train(
    data='../../dataset/04_all_classes/data.yaml',
    
    # --- OPTIMIERUNGS-SETUP ---
    epochs=40,            # Reicht völlig aus, um den Nano-Run zu schlagen
    imgsz=960,            # 960px ist der Sweetspot: Groß genug für Achsenbeschriftungen,
                          # aber klein genug, um schnell zu trainieren.
    
    # Mit Small Modell und 24GB VRAM können wir die Batch-Size etwas hochdrehen
    batch=24,             
    
    name='compound_yolo11s_960_optimized',
    device=0,
    patience=10,
    
    # --- NOTEBOOK STABILITY ---
    workers=8,            # Verhindert Crash
    cache=True,           # RAM-Beschleunigung an (macht workers=0 wett)
    exist_ok=True
)

Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=compound_yolo11s_960_optimized, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

### Train YOLO on the stratified, oversampled dataset with additional synthetic plots

In [ ]:
from ultralytics import YOLO

# Wähle das Modell:
# 'yolo11n.pt' -> Nano (Schnellster Test)
# 'yolo11m.pt' -> Medium (Gute Balance für später)
# 'yolo11l.pt' -> Large (Hohe Genauigkeit für 3090)
MODEL_VARIANT = 'yolo11n.pt' 

print(f"Lade Modell: {MODEL_VARIANT}...")
model = YOLO(MODEL_VARIANT)

# Training starten
results = model.train(
    data='../../dataset/04_all_classes/data.yaml', # Pfad muss stimmen!
    epochs=20,          # Für Nano Test reichen 50. Für Large später 100+
    imgsz=640,          # Standard. Bei 3090 könntest du auch 1280 testen (High Res)
    batch=32,           # 3090 packt bei Nano locker 64/128, bei Large ca 16/24
    name='compound_yolo11n_sim2real',
    device=0,           # Erzwingt GPU 0
    workers=8,          # Schnelleres Laden der Daten
    exist_ok=True       # Überschreibt den Ordner falls er existiert (gut zum Testen)
)

Lade Modell: yolo11n.pt...
New https://pypi.org/project/ultralytics/8.3.242 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=F

In [ ]:
from ultralytics import YOLO

# Wähle das Modell:
# 'yolo11n.pt' -> Nano (Schnellster Test)
# 'yolo11s.pt' -> Small (Guter Kompromiss)
# 'yolo11m.pt' -> Medium (Gute Balance für später)
# 'yolo11l.pt' -> Large (Hohe Genauigkeit für 3090)
MODEL_VARIANT = 'yolo11s.pt' 

print(f"Lade Modell: {MODEL_VARIANT}...")
model = YOLO(MODEL_VARIANT)

# Training starten
results = model.train(
    data='../../dataset/04_all_classes/data.yaml', # Pfad muss stimmen!
    epochs=40,          # Für Nano Test reichen 50. Für Large später 100+
    imgsz=960,          # Standard. Bei 3090 könntest du auch 1280 testen (High Res)
    batch=16,           # 3090 packt bei Nano locker 64/128, bei Large ca 16/24
    name='compound_yolo11s_sim2real',
    device=0,           # Erzwingt GPU 0
    workers=8,          # Schnelleres Laden der Daten
    exist_ok=True,       # Überschreibt den Ordner falls er existiert (gut zum Testen)
    
    # Ein bisschen Augmentation gegen Overfitting auf Synth-Style
    mixup=0.1,       
    copy_paste=0.1
)

Lade Modell: yolo11s.pt...
New https://pypi.org/project/ultralytics/8.3.242 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=F

In [ ]:
from ultralytics import YOLO

# Wir wechseln auf MEDIUM für mehr "Verständnis" von Layouts
model = YOLO('../../models/yolo11m.pt') 

results = model.train(
    data='../../dataset/04_all_classes/data.yaml',
    
    # --- PFLICHT-UPGRADES ---
    epochs=80,             # Gib ihm Zeit. Die ersten 20 Epochen sind nur Aufwärmen.
    imgsz=1280,            # Absolutes Muss für Achsen-Texte.
    batch=8,               # 1280px frisst VRAM. Starte mit 8. Wenn OOM, geh auf 4.
    
    # --- PERFORMANCE & OPTIMIERUNG ---
    name='compound_FINAL_medium_1280',
    device=0,
    workers=8,       
    
    # --- AUGMENTATION ---
    # Wir machen es dem Modell etwas schwerer, damit es robuster wird
    mosaic=1.0, 
    mixup=0.15,            # Leicht erhöht
    copy_paste=0.15,       # Leicht erhöht
    
    patience=20,           # Early Stopping
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.242 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=compound_FINAL

In [ ]:
from ultralytics import YOLO

# Pfad zu deinem besten Modell aus dem abgebrochenen Run
# WICHTIG: Prüfe den Pfad!
model_path = '/home/tobiasponeschtu/Documents/11818774_CompoundFigureSeparation/runs/detect/compound_FINAL_medium_1280/weights/best.pt'
model = YOLO(model_path)

print("Starte Smart Fine-Tuning (Warmup -> Precision)...")

results = model.train(
    data='../../dataset/04_all_classes/data.yaml',
    
    # --- STRATEGIE ---
    epochs=30,             # Insgesamt 30 Epochen
    close_mosaic=20,       # WICHTIG: Die letzten 20 Epochen sind OHNE Mosaic!
                           # (D.h. Epoche 1-10 sind MIT Mosaic)
    
    imgsz=1280,
    batch=8,
    
    name='compound_FINAL_medium_1280_SMART_FINETUNE',
    device=0,
    workers=8,
    
    # --- HYPERPARAMETER TWEAKS ---
    lr0=0.005,             # Wir starten mit halber Power (Standard ist 0.01)
    lrf=0.1,               # End-Lernrate sinkt sanft ab
    
    # Verhindert, dass er sofort wieder abbricht
    patience=0,            # Early Stopping komplett AUS. Wir ziehen das durch.
    exist_ok=True,
    
    # Ein bisschen Mixup hilft in der ersten Phase gegen Overfitting
    mixup=0.1 
)

Starte Smart Fine-Tuning (Warmup -> Precision)...
New https://pypi.org/project/ultralytics/8.3.243 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/home/tobiasponeschtu/Document

In [ ]:
from ultralytics import YOLO

# Neues, frisches Small-Modell laden
model = YOLO('yolo11s.pt') 

results = model.train(
    data='../../dataset/04_all_classes/data.yaml',
    
    # --- VERGLEICHS-SETUP ---
    epochs=60,             # 60 reichen für den Vergleich
    imgsz=1280,            # High Res beibehalten!
    
    # Hier ist der Vorteil: Wir erhöhen die Batch Size
    batch=16,              # Sollte auf 24GB VRAM passen. Falls OOM -> 16.
    
    name='comparison_small_1280_batch16',
    device=0,
    workers=8,
    
    # Augmentation (Standard lassen oder wie bei Medium)
    patience=0,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.243 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/04_model_ready/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=comparison_smal

## Training YOLO on the selected classes set

In [1]:
from ultralytics import YOLO

DATASET_YAML = '../../dataset/05_selected_classes/data.yaml'

# Gemeinsame Einstellungen für alle Runs
COMMON_PARAMS = {
    'data': DATASET_YAML,
    'epochs': 50,
    'imgsz': 1280,
    'device': 0,
    'workers': 8,
    'patience': 0,
    'exist_ok': True,      
    'close_mosaic': 10,
    'project': 'runs/selected_classes',
}

WRITE SMTH HERE!

### YOLO Nano 

In [2]:
model = YOLO('../../models/yolo11n.pt') 

results = model.train(
    **COMMON_PARAMS,
    name='compound_figure_separator_selected_classes_yolo11n_1280_batch16',
    batch=16,
)

New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/05_selected_classes/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, na

### YOLO Small 

In [ ]:
from ultralytics import YOLO

# Neues, frisches Small-Modell laden
model = YOLO('../../models/yolo11s.pt') 

results = model.train(
    data='../../dataset/05_selected_classes/data.yaml',
    
    # --- VERGLEICHS-SETUP ---
    epochs=50,             # 50 reichen für den Vergleich
    imgsz=1280,            # High Res beibehalten!
    
    # Hier ist der Vorteil: Wir erhöhen die Batch Size
    batch=16,              # Sollte auf 24GB VRAM passen. Falls OOM -> 16.
    
    name='comparison_small_1280_batch16',
    device=0,
    workers=8,
    
    # Augmentation (Standard lassen oder wie bei Medium)
    patience=0,
    exist_ok=True
)

### YOLO Medium 

## Training YOLO on the selected classes set

In [1]:
from ultralytics import YOLO

DATASET_YAML = '../../dataset/06_compound_chart_splitter/data.yaml'

# Gemeinsame Einstellungen für alle Runs
COMMON_PARAMS = {
    'data': DATASET_YAML,
    'epochs': 50,
    'imgsz': 1280,
    'device': 0,
    'workers': 8,
    'patience': 0,
    'exist_ok': True,      
    'close_mosaic': 10,
    'project': 'runs/compound_chart_splitter',
}

In [2]:
model = YOLO('../../models/yolo11n.pt') 

results = model.train(
    **COMMON_PARAMS,
    name='compound_chart_splitter_yolo11n_1280_batch16',
    batch=16,
)

New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/06_compound_chart_splitter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=Fa

In [2]:
model = YOLO('../../models/yolo11s.pt') 

results = model.train(
    **COMMON_PARAMS,
    name='compound_chart_splitter_yolo11s_1280_batch16',
    batch=16,
)

New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/06_compound_chart_splitter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../models/yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=Fa

In [1]:
from ultralytics import YOLO

DATASET_YAML = '../../dataset/06_compound_chart_splitter/data.yaml'

# Gemeinsame Einstellungen für alle Runs
COMMON_PARAMS = {
    'data': DATASET_YAML,
    'epochs': 100,
    'imgsz': 1280,
    'device': 0,
    'workers': 4,
    'patience': 0,
    'exist_ok': True,      
    'close_mosaic': 10,
    'project': 'runs/compound_chart_splitter',
}

model = YOLO('../../models/yolo11m.pt') 

results = model.train(
    **COMMON_PARAMS,
    name='compound_chart_splitter_yolo11m_1280_batch8',
    batch=8,
)

New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/06_compound_chart_splitter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../models/yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=Fa

In [1]:
from ultralytics import YOLO
import torch
import gc

# 1. Cleanup: VRAM freimachen
gc.collect()
torch.cuda.empty_cache()

DATASET_YAML = '../../dataset/06_compound_chart_splitter/data.yaml'

# --- OPTIMIERTE RUN CONFIG ---
COMMON_PARAMS = {
    'data': DATASET_YAML,
    'epochs': 100,
    'imgsz': 1280,
    'device': 0,
    'workers': 4,
    'exist_ok': True,
    'project': 'runs/compound_chart_splitter',
    
    # --- TUNING-UPDATE ---
    
    # 1. Geduld: Erhöht, damit er bei kleinen Plateaus nicht abbricht
    'patience': 20,         
    
    # 2. Mosaic: Längere "Clean Phase" am Ende
    'close_mosaic': 20,     # Letzte 20 Epochen nur echte Bilder (hilft Layout zu lernen)
    
    # 3. Augmentation: Text schützen! (Das ist der Schlüssel für Title/Legend)
    'degrees': 0.0,         # 0.0 = Text niemals drehen (vorher Standard)
    'perspective': 0.0,     # 0.0 = Keine Verzerrung
    'shear': 0.0,           # 0.0 = Keine Scherung
    'scale': 0.2,           # 0.2 = Weniger aggressiver Zoom (Textgröße bleibt stabiler)
    'flipud': 0.0,          # 0.0 = Niemals auf den Kopf stellen (Text!)
    
    # 4. Fokus verschieben
    'cls': 2.0,             # Klassifizierungs-Loss verdoppelt (hilft Legend vs Title zu trennen)
    'optimizer': 'AdamW',   # AdamW hilft oft bei feineren Strukturen
    'lr0': 0.001,           # Sanftere Lernrate
    'cos_lr': True          # Sanftes Auslaufen
}

# Medium Modell laden
model = YOLO('../../models/yolo11m.pt') 

print("🚀 STARTE TUNED TRAINING (No-Rotation, High-Cls-Loss)...")

results = model.train(
    **COMMON_PARAMS,
    name='compound_chart_splitter_yolo11m_1280_tuned_v1',
    batch=8,
)

🚀 STARTE TUNED TRAINING (No-Rotation, High-Cls-Loss)...
New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=20, cls=2.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../../dataset/06_compound_chart_splitter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../../model

In [1]:
from ultralytics import YOLO
import torch
import gc

# VRAM Cleanup
gc.collect()
torch.cuda.empty_cache()

DATASET_YAML = '../../dataset/06_compound_chart_splitter/data.yaml'

COMMON_PARAMS = {
    'data': DATASET_YAML,
    'epochs': 50,
    'imgsz': 1280,
    'device': 0,
    'workers': 6,
    'patience': 0,          # 0 = Deaktiviert Early Stopping, erzwingt alle 100 Epochen
    'exist_ok': True,
    'close_mosaic': 10,
    'project': 'runs/compound_chart_splitter',
    'cache': False,
}

model = YOLO('yolo11l.pt')

results = model.train(
    **COMMON_PARAMS,
    name='compound_chart_splitter_yolo11l_1280_batch8',
    batch=8,
)

New https://pypi.org/project/ultralytics/8.3.246 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24112MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../dataset/06_compound_chart_splitter/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=comp